# 3. Process MSA listings

Parses **every** raw snapshot in `data/raw/msa/` and rebuilds the outputs from scratch each run (never appends), so fixing a parser or a mapping in `mappings.py` corrects the whole history.

**Outputs**
- `outputs/msa_master.csv`: long format, full history. One row per snapshot date × manufacturer × product × state × utility. Fields are kept as the manufacturer wrote them, with standardized `*_std` columns alongside. Nothing is filtered.
- `outputs/msa_summary.csv`: the current picture. One row per state × utility with a Tesla / Enphase / ConnectDER status column (each brand's best status across its products, from its latest snapshot; blank = not approved).

In [ ]:
import json
import re

import pandas as pd
from bs4 import BeautifulSoup

from common import MSA_MASTER_CSV, MSA_SOURCE_PAGES, MSA_SUMMARY_CSV, RAW_MSA_DIR, get_logger
from mappings import (APPROVAL_STD, BRANDS, ENPHASE_LIMITATION_PILLS, STATE_NAMES, STATE_STD,
                      SUMMARY_ORDER, SUMMARY_STATUS, UTILITY_STD)

log = get_logger("process_msa")


def clean(text):
    """Collapse whitespace and drop zero-width characters (one ConnectDER
    name carries a trailing U+200B)."""
    text = re.sub(r"[\u200b-\u200d\ufeff]", "", text or "")
    return re.sub(r"\s+", " ", text).strip()

## Parsers

One per manufacturer. Each takes a snapshot folder and returns a list of rows, or `None` if that source wasn't fetched in that snapshot. Standardized columns are added afterwards in one place.

In [ ]:
def parse_connectder(snap_dir):
    """Prismic JSON: one utility_provider document per utility, with one
    product_statuses entry per state it serves, each holding three products."""
    docs = []
    for path in sorted(snap_dir.glob("connectder_p*.json")):
        docs += json.loads(path.read_text())["results"]
    if not docs:
        return None

    install_types = {d["id"]: d["data"]["label"] for d in docs if d["type"] == "utility_install_type"}
    products = {"islandder": "IslandDER MSA", "solar_msa": "Solar MSA", "ev_msa": "EV MSA"}
    rows = []
    for doc in docs:
        if doc["type"] != "utility_provider":
            continue
        data = doc["data"]
        other_names = "; ".join(o["other_name"] for o in data.get("other_names") or [] if o.get("other_name"))
        for ps in data["product_statuses"]:
            install_type = install_types.get((ps.get("utility_install_type") or {}).get("id"), "")
            for key, product in products.items():
                rows.append({
                    "manufacturer": "ConnectDER", "product": product,
                    "state": ps.get("state") or "", "utility": clean(data["name"]),
                    "utility_other_names": other_names,
                    "approval_raw": ps.get(f"{key}_status") or "",
                    "expected_date": ps.get(f"{key}_due_date") or "",
                    "install_type": install_type, "notes": "",
                })
    return rows


def parse_enphase(snap_dir):
    """HTML. The page lists utilities twice (by state and by utility); only the
    'Sort by state' tab is read. Status pills become the approval level, except
    limitation pills (e.g. ring-type bases only), which go in notes."""
    path = snap_dir / "enphase.html"
    if not path.exists():
        return None
    tab = BeautifulSoup(path.read_text(), "html.parser").select_one('.tabs__item[data-tab="Sort by state"]')
    if tab is None:
        raise ValueError("'Sort by state' tab not found; page layout changed")

    rows = []
    for state_block in tab.select(".state-accordion"):
        state = clean(state_block.select_one(".state-accordion__state-name").get_text())
        for item in state_block.select("li.state-accordion__list-item"):
            pill = item.select_one(".state-accordion__status-pill")
            pill_text = clean(pill.get_text()) if pill else ""
            if pill:
                pill.extract()
            is_limitation = pill_text in ENPHASE_LIMITATION_PILLS
            rows.append({
                "manufacturer": "Enphase", "product": "IQ Meter Collar",
                "state": state, "utility": clean(item.get_text()),
                "approval_raw": "Listed" if not pill_text or is_limitation else pill_text,
                "notes": pill_text if is_limitation else "",
            })
    return rows


def parse_tesla(snap_dir):
    """HTML accordions. Two lists, Backup Switch approved for Powerwall and for
    Powershare installs, each under its own h3 heading."""
    path = snap_dir / "tesla.html"
    if not path.exists():
        return None
    soup = BeautifulSoup(path.read_text(), "html.parser")

    rows = []
    for accordion in soup.select("section.tcl-accordion"):
        heading = clean(accordion.find_previous("h3").get_text())
        match = re.search(r"for (Powerwall|Powershare) Installation", heading)
        if not match:
            raise ValueError(f"unexpected heading above accordion: {heading!r}")
        for item in accordion.select(".tcl-accordion__item"):
            state = clean(item.select_one(".tcl-accordion__title").get_text())
            for li in item.select(".tcl-accordion__panel li"):
                rows.append({
                    "manufacturer": "Tesla", "product": f"Backup Switch ({match.group(1)})",
                    "state": state, "utility": clean(li.get_text()),
                    "approval_raw": "Listed", "notes": "",
                })
    if not rows:
        raise ValueError("no utilities found; page layout changed")
    return rows


PARSERS = {"ConnectDER": parse_connectder, "Enphase": parse_enphase, "Tesla": parse_tesla}

## Parse all snapshots

In [ ]:
snapshots = sorted(p for p in RAW_MSA_DIR.glob("????-??-??") if p.is_dir())
log.info("start  %d snapshot(s)", len(snapshots))

frames, errors = [], 0
for snap_dir in snapshots:
    counts = []
    for manufacturer, parse in PARSERS.items():
        try:
            rows = parse(snap_dir)
        except Exception as e:
            errors += 1
            log.error("FAILED  %s %-10s %s: %s", snap_dir.name, manufacturer, type(e).__name__, e)
            continue
        if rows is None:
            counts.append(f"{manufacturer}=not fetched")
            continue
        frames.append(pd.DataFrame(rows).assign(snapshot_date=snap_dir.name,
                                                source_url=MSA_SOURCE_PAGES[manufacturer]))
        counts.append(f"{manufacturer}={len(rows)}")
    log.info("parsed  %s  %s", snap_dir.name, "  ".join(counts))

## Standardize → `msa_master.csv`

In [ ]:
COLUMNS = ["snapshot_date", "manufacturer", "product", "state", "state_std", "utility", "utility_std",
           "utility_other_names", "approval_raw", "approval_std", "expected_date", "install_type",
           "notes", "source_url"]

master = pd.concat(frames, ignore_index=True).reindex(columns=COLUMNS).fillna("")
master["state_std"] = master["state"].map(STATE_STD).fillna("")
master["utility_std"] = master["utility"].map(lambda u: UTILITY_STD.get(u, u))
master["approval_std"] = master["approval_raw"].map(APPROVAL_STD).fillna("UNMAPPED")

for col, std in [("approval_raw", "approval_std"), ("state", "state_std")]:
    # A blank state is verbatim from the source, not a missing mapping.
    unmapped = master.loc[master[std].isin(["", "UNMAPPED"]) & (master[col] != ""), col].unique()
    if len(unmapped):
        log.warning("unmapped %s values (add to mappings.py): %s", col, sorted(unmapped))

master = master.sort_values(["snapshot_date", "manufacturer", "product", "state", "utility"])
master.to_csv(MSA_MASTER_CSV, index=False)
log.info("wrote %s: %d rows", MSA_MASTER_CSV.name, len(master))
master.groupby(["snapshot_date", "manufacturer", "product", "approval_std"]).size().unstack(fill_value=0)

## Current status per brand → `msa_summary.csv`

In [ ]:
# Each brand's latest snapshot (a catch-up retry can put one brand in a later folder).
latest = master.groupby("manufacturer")["snapshot_date"].transform("max")
current = master[(master["snapshot_date"] == latest) & (master["state_std"] != "")].copy()
current["status"] = current["approval_std"].map(SUMMARY_STATUS).fillna("UNMAPPED")
current["rank"] = current["status"].map({s: i for i, s in enumerate(SUMMARY_ORDER)}).fillna(len(SUMMARY_ORDER))

# Best status per brand across its products.
best = current.sort_values("rank").drop_duplicates(["state_std", "utility_std", "manufacturer"])
summary = (best.pivot(index=["state_std", "utility_std"], columns="manufacturer", values="status")
           .reindex(columns=BRANDS).fillna("").reset_index())
summary.columns.name = None
summary.insert(0, "state", summary.pop("state_std").map(STATE_NAMES))
summary = (summary.rename(columns={"utility_std": "utility"})
           .sort_values(["state", "utility"], key=lambda c: c.str.lower()))

summary.to_csv(MSA_SUMMARY_CSV, index=False)
log.info("wrote %s: %d state x utility rows (latest snapshots: %s)", MSA_SUMMARY_CSV.name, len(summary),
         ", ".join(f"{m} {d}" for m, d in master.groupby("manufacturer")["snapshot_date"].max().items()))
pd.DataFrame({b: summary[b].replace("", "(blank)").value_counts() for b in BRANDS}).fillna(0).astype(int)

In [ ]:
if errors:
    raise RuntimeError(f"{errors} parser error(s); see run_log.txt")